In [1]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from tkinter import Tk, filedialog
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
root = Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = filedialog.askopenfilename(
    title="Select your cleaned e-commerce Excel dataset",
    filetypes=[
        ("Excel files", "*.xlsx *.xls"),
        ("All files", "*.*")
    ]
)

root.destroy()

if not file_path:
    raise ValueError("No file selected. Please select your Excel dataset.")

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("File:", Path(file_path).name)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
File: ecommerce_data_analysis.xlsx
Rows: 5
Columns: 2


,Metric,Value
0,Total Sales,4.915545e+06
1,Average Order Value,9.831089e+02
2,Total Unique Orders,5.000000e+03
3,Total Unique Customers,5.000000e+03
4,Total Quantity Sold,1.110000e+04


In [4]:
original_df = df.copy(deep=True)

print("Original dataset backed up in memory.")
print("Original shape:", original_df.shape)

Original dataset backed up in memory.
Original shape: (5, 2)


In [5]:
print("Dataset Information:")
print("-" * 50)

print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame(name="Data_Type"))

print("\nFirst 10 rows:")
display(df.head(10))

print("\nLast 5 rows:")
display(df.tail())

print("\nStatistical summary:")
display(df.describe(include="all").T)

Dataset Information:
--------------------------------------------------
Rows and columns: (5, 2)

Column names:
['Metric', 'Value']

Data types:


,Data_Type
Metric,str
Value,float64



First 10 rows:


,Metric,Value
0,Total Sales,4.915545e+06
1,Average Order Value,9.831089e+02
2,Total Unique Orders,5.000000e+03
3,Total Unique Customers,5.000000e+03
4,Total Quantity Sold,1.110000e+04



Last 5 rows:


,Metric,Value
0,Total Sales,4.915545e+06
1,Average Order Value,9.831089e+02
2,Total Unique Orders,5.000000e+03
3,Total Unique Customers,5.000000e+03
4,Total Quantity Sold,1.110000e+04



Statistical summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Metric,5,5,Total Sales,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Value,5.0,NaN,NaN,NaN,987525.535783,2195832.369344,983.108914,5000.0,5000.0,11100.0,4915544.57


In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

print("Column names standardized:")
print(df.columns.tolist())

Column names standardized:
['Metric', 'Value']


In [7]:
missing_before = df.isna().sum()

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Values": missing_before.values,
    "Missing_Percentage": (
        missing_before.values / len(df) * 100
    ).round(2)
})

print("Missing-value report:")
display(missing_report)

print("Total missing cells:", int(missing_before.sum()))

Missing-value report:


,Column,Missing_Values,Missing_Percentage
0,Metric,0,0.0
1,Value,0,0.0


Total missing cells: 0


In [8]:
rows_before = len(df)
duplicate_rows = int(df.duplicated().sum())

df = df.drop_duplicates().copy()

rows_after = len(df)
duplicates_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Duplicate rows found:", duplicate_rows)
print("Duplicate rows removed:", duplicates_removed)
print("Rows after:", rows_after)

Rows before: 5
Duplicate rows found: 0
Duplicate rows removed: 0
Rows after: 5


In [10]:
id_columns = ["Order_ID", "Customer_ID"]

for column in id_columns:
    if column in df.columns:
        duplicate_count = int(
            df[column].dropna().duplicated().sum()
        )

        print(f"{column}: {duplicate_count} repeated non-empty IDs")

        if duplicate_count > 0:
            display(
                df[df[column].duplicated(keep=False)]
                .sort_values(column)
            )

In [11]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    df[column] = df[column].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

print("Removed leading and trailing spaces from text fields.")

Removed leading and trailing spaces from text fields.


In [12]:
if "Product_Category" in df.columns:
    df["Product_Category"] = df["Product_Category"].replace({
        "sports": "Sports"
    })

print("Product categories after standardization:")

if "Product_Category" in df.columns:
    display(
        df["Product_Category"]
        .value_counts(dropna=False)
        .rename_axis("Product_Category")
        .reset_index(name="Count")
    )

Product categories after standardization:


In [13]:
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(
        df["Date"], errors="coerce"
    )

numeric_columns = [
    "Age",
    "Unit_Price",
    "Quantity",
    "Discount_Amount",
    "Total_Amount",
    "Session_Duration_Minutes",
    "Pages_Viewed",
    "Delivery_Time_Days",
    "Customer_Rating"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column], errors="coerce"
        )

integer_columns = [
    "Age",
    "Quantity",
    "Session_Duration_Minutes",
    "Pages_Viewed",
    "Delivery_Time_Days",
    "Customer_Rating"
]

for column in integer_columns:
    if column in df.columns:
        non_missing = df[column].dropna()

        if (non_missing % 1 == 0).all():
            df[column] = df[column].astype("Int64")

if "Is_Returning_Customer" in df.columns:
    df["Is_Returning_Customer"] = (
        df["Is_Returning_Customer"]
        .astype("boolean")
    )

print("Data type conversion completed.")
display(df.dtypes.to_frame(name="Data_Type"))

Data type conversion completed.


,Data_Type
Metric,str
Value,float64


In [14]:
missing_after_conversion = df.isna().sum()

print("Missing values after data type conversion:")
display(
    missing_after_conversion[
        missing_after_conversion > 0
    ].rename("Missing_Count").to_frame()
)

print(
    "Total missing cells after conversion:",
    int(missing_after_conversion.sum())
)

Missing values after data type conversion:


,Missing_Count


Total missing cells after conversion: 0


In [15]:
range_rules = {
    "Age": (18, 100),
    "Unit_Price": (0, np.inf),
    "Quantity": (1, np.inf),
    "Discount_Amount": (0, np.inf),
    "Total_Amount": (0, np.inf),
    "Session_Duration_Minutes": (0, np.inf),
    "Pages_Viewed": (1, np.inf),
    "Delivery_Time_Days": (0, np.inf),
    "Customer_Rating": (1, 5)
}

range_results = []

for column, (minimum, maximum) in range_rules.items():
    if column not in df.columns:
        continue

    invalid_mask = (
        df[column].notna()
        & (
            (df[column] < minimum)
            | (df[column] > maximum)
        )
    )

    range_results.append({
        "Column": column,
        "Minimum_Allowed": minimum,
        "Maximum_Allowed": maximum,
        "Invalid_Count": int(invalid_mask.sum())
    })

range_report = pd.DataFrame(range_results)

print("Range validation report:")
display(range_report)

Range validation report:


""


In [16]:
required_columns = [
    "Unit_Price",
    "Quantity",
    "Discount_Amount",
    "Total_Amount"
]

if all(column in df.columns for column in required_columns):
    df["Expected_Total_Amount"] = (
        df["Unit_Price"] * df["Quantity"]
        - df["Discount_Amount"]
    ).round(2)

    df["Amount_Check"] = np.isclose(
        df["Total_Amount"],
        df["Expected_Total_Amount"],
        atol=0.02,
        rtol=0,
        equal_nan=False
    )

    mismatch_count = int((~df["Amount_Check"]).sum())

    print("Order amount mismatches:", mismatch_count)

    if mismatch_count:
        display(
            df.loc[
                ~df["Amount_Check"],
                [
                    "Order_ID",
                    "Unit_Price",
                    "Quantity",
                    "Discount_Amount",
                    "Total_Amount",
                    "Expected_Total_Amount"
                ]
            ].head(20)
        )
else:
    print("Required amount columns are not available.")

Required amount columns are not available.


In [18]:
outlier_columns = [
    "Age",
    "Unit_Price",
    "Quantity",
    "Discount_Amount",
    "Total_Amount",
    "Session_Duration_Minutes",
    "Pages_Viewed",
    "Delivery_Time_Days",
    "Customer_Rating"
]

outlier_results = []

for column in outlier_columns:
    if column not in df.columns:
        continue

    values = df[column].dropna()

    if len(values) < 2:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (df[column] < lower_bound)
        | (df[column] > upper_bound)
    )

    outlier_results.append({
        "Column": column,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "Lower_Bound": round(lower_bound, 2),
        "Upper_Bound": round(upper_bound, 2),
        "Outlier_Count": int(outlier_mask.sum())
    })

outlier_report = pd.DataFrame(outlier_results)

print("IQR outlier detection report:")
display(outlier_report)

IQR outlier detection report:


""


In [19]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Number of rows",
        "Number of columns",
        "Missing cells",
        "Duplicate full rows",
        "Duplicate Order_ID values",
        "Duplicate Customer_ID values"
    ],
    "Before_Cleaning": [
        len(original_df),
        len(original_df.columns),
        int(original_df.isna().sum().sum()),
        int(original_df.duplicated().sum()),
        (
            int(original_df["Order_ID"].dropna().duplicated().sum())
            if "Order_ID" in original_df.columns else None
        ),
        (
            int(original_df["Customer_ID"].dropna().duplicated().sum())
            if "Customer_ID" in original_df.columns else None
        )
    ],
    "After_Cleaning": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        (
            int(df["Order_ID"].dropna().duplicated().sum())
            if "Order_ID" in df.columns else None
        ),
        (
            int(df["Customer_ID"].dropna().duplicated().sum())
            if "Customer_ID" in df.columns else None
        )
    ]
})

print("Before-and-after quality summary:")
display(quality_summary)

Before-and-after quality summary:


,Metric,Before_Cleaning,After_Cleaning
0,Number of rows,5.0,5.0
1,Number of columns,2.0,2.0
2,Missing cells,0.0,0.0
3,Duplicate full rows,0.0,0.0
4,Duplicate Order_ID values,NaN,NaN
5,Duplicate Customer_ID values,NaN,NaN


In [20]:
final_missing_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    ).round(2)
})

print("Final missing-value report:")
display(final_missing_report)

Final missing-value report:


,Column,Data_Type,Missing_Count,Missing_Percentage
0,Metric,str,0,0.0
1,Value,float64,0,0.0


In [21]:
data_dictionary = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": [str(df[col].dtype) for col in df.columns],
    "Description": [
        {
            "Order_ID": "Unique identifier for an order",
            "Customer_ID": "Identifier for the customer",
            "Date": "Date of the order",
            "Age": "Customer age",
            "Gender": "Customer gender category",
            "City": "Customer city",
            "Product_Category": "Category of purchased product",
            "Unit_Price": "Price per unit",
            "Quantity": "Number of units ordered",
            "Discount_Amount": "Discount applied to the order",
            "Total_Amount": "Recorded total order amount",
            "Payment_Method": "Method used for payment",
            "Device_Type": "Device used by the customer",
            "Session_Duration_Minutes": "Duration of the customer session",
            "Pages_Viewed": "Number of pages viewed",
            "Is_Returning_Customer": "Whether the customer is returning",
            "Delivery_Time_Days": "Delivery duration in days",
            "Customer_Rating": "Customer rating from 1 to 5",
            "Expected_Total_Amount": "Calculated amount using price, quantity and discount",
            "Amount_Check": "Whether recorded and calculated totals agree"
        }.get(col, "Column in the uploaded dataset")
        for col in df.columns
    ]
})

print("Data dictionary created:")
display(data_dictionary)

Data dictionary created:


,Column_Name,Data_Type,Description
0,Metric,str,Column in the uploaded dataset
1,Value,float64,Column in the uploaded dataset


In [22]:
cleaned_data = df.drop(
    columns=[
        "Expected_Total_Amount",
        "Amount_Check"
    ],
    errors="ignore"
).copy()

print("Final dataset prepared.")
print("Rows:", cleaned_data.shape[0])
print("Columns:", cleaned_data.shape[1])

display(cleaned_data.head(10))

Final dataset prepared.
Rows: 5
Columns: 2


,Metric,Value
0,Total Sales,4.915545e+06
1,Average Order Value,9.831089e+02
2,Total Unique Orders,5.000000e+03
3,Total Unique Customers,5.000000e+03
4,Total Quantity Sold,1.110000e+04


In [24]:
if "output_path" in globals() and Path(output_path).exists():
    check = pd.ExcelFile(output_path)

    print("File verified successfully!")
    print("Saved file:", Path(output_path).name)
    print("Sheets available:")

    for sheet in check.sheet_names:
        print("-", sheet)

    verification_df = pd.read_excel(
        output_path,
        sheet_name="Cleaned_Data"
    )

    print("\nFinal dataset dimensions:", verification_df.shape)
    display(verification_df.head())
else:
    print("No saved workbook found. Run Cell 22 first.")

File verified successfully!
Saved file: cleaned dataset by using vs code1.xlsx
Sheets available:
- Cleaned_Data
- Quality_Summary
- Missing_Values
- Range_Checks
- Outlier_Report
- Data_Dictionary

Final dataset dimensions: (5, 2)


,Metric,Value
0,Total Sales,4.915545e+06
1,Average Order Value,9.831089e+02
2,Total Unique Orders,5.000000e+03
3,Total Unique Customers,5.000000e+03
4,Total Quantity Sold,1.110000e+04
